In [1]:
import sys
import os
import glob
import librosa
import numpy as np

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf

In [2]:
def file_to_vector_array(file_name, n_mels=128, frames=32, n_fft=1024, hop_length=512, power=2.0):
    y, sr = librosa.load(file_name, sr=None, mono=True)
    mel_spectrogram = librosa.feature.melspectrogram(y=y,
                                                     sr=sr,
                                                     n_fft=n_fft,
                                                     hop_length=hop_length,
                                                     n_mels=n_mels,
                                                     power=power)

    log_mel_spectrogram = librosa.power_to_db(mel_spectrogram, ref=np.max).astype(np.float32)
    vector_array = log_mel_spectrogram.T

    return vector_array

def make_windows(seq, window = 128, hop = 32):
    T, F = seq.shape
    if T < window:
        pad = np.repeat(seq[-1:, :], repeats=(window - T), axis=0)
        seq = np.concatenate([seq, pad], axis=0)
        T = window

    windows = []
    for start in range(0, T - window + 1, hop):
        windows.append(seq[start:start + window])
    return np.stack(windows, axis=0).astype(np.float32)

In [3]:
def list_wavs_by_machine_id(train_dir, machine_id):
    training_list_path = os.path.relpath("../dataset/{dir}/*_id_{id}_*.wav".format(dir=train_dir, id=machine_id))
    files = sorted(glob.glob(training_list_path))
    return files

In [4]:
from tensorflow import keras
from tensorflow.keras import layers

def build_seq2seq_lstm_ae(timesteps: int, n_features: int = 128, latent_dim: int = 32, units: int = 128, dropout: float = 0.1):
    inputs = keras.Input(shape=(timesteps, n_features))
    x = layers.LSTM(units, return_sequences=False, dropout=dropout)(inputs)
    z = layers.Dense(latent_dim, activation="relu", name="latent")(x)

    x = layers.RepeatVector(timesteps)(z)
    x = layers.LSTM(units, return_sequences=True, dropout=dropout)(x)
    outputs = layers.TimeDistributed(layers.Dense(n_features))(x)

    model = keras.Model(inputs, outputs, name="logmel_seq2seq_lstm_ae")
    return model

In [5]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import StandardScaler
from keras.callbacks import EarlyStopping, ModelCheckpoint

machine_config = {
    'fan': ['00', '02', '04', '06'],
    'valve': ['00', '02', '04', '06'],
    'pump': ['00', '02', '04', '06'],
    'slider': ['00', '02', '04', '06'],
    'ToyCar': ['01', '02', '03', '04'],
    'ToyConveyor': ['01', '02', '03']
}

lstm_ae_results = []

for machine_type, machine_ids in machine_config.items():
    for m_id in machine_ids:
        print(f'--- (LSTM AE) Processing {machine_type} with ID: {m_id} ---')

        train_files = list_wavs_by_machine_id(f'{machine_type}/train', m_id)

        train_files, val_files = train_test_split(train_files, test_size=0.1, random_state=42)

        all_w = []
        for file in train_files:
            seq = file_to_vector_array(file)
            w = make_windows(seq)
            all_w.append(w)

        train_data = np.concatenate(all_w, axis=0).astype(np.float32)

        mean = train_data.reshape(-1, train_data.shape[-1]).mean(axis=0)
        std = train_data.reshape(-1, train_data.shape[-1]).std(axis=0) + 1e-7

        train_data = (train_data - mean) / std

        all_w = []
        for file in val_files:
            seq = file_to_vector_array(file)
            w = make_windows(seq)
            all_w.append(w)

        val_data = np.concatenate(all_w, axis=0).astype(np.float32)

        val_data = (val_data - mean) / std

        callbacks = [
            EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
            ModelCheckpoint(filepath=f'../models/lstm_ae/lstm_ae_model_{machine_type}_{m_id}.keras', monitor='val_loss', mode='min', save_best_only=True)
        ]

        timesteps, n_features = train_data.shape[1], train_data.shape[2]

        model = build_seq2seq_lstm_ae(timesteps=timesteps, n_features=n_features, latent_dim=32, units=128)
        model.compile(optimizer="adam", loss="mse")

        model.fit(
            train_data, train_data,
            epochs=100,
            batch_size=256,
            validation_data=(val_data, val_data),
            callbacks=callbacks
        )

        test_files = list_wavs_by_machine_id(f'{machine_type}/test', m_id)
        true_labels = [1 if 'anomaly' in os.path.basename(f) else 0 for f in test_files]

        y_pred = []
        for file in test_files:
            seq = file_to_vector_array(file)
            X_windows = make_windows(seq)
            X_norm = (X_windows - mean) / std
            X_hat = model.predict(X_norm, verbose=0)
            errors = np.mean((X_norm - X_hat) ** 2, axis=(1, 2))
            y_pred.append(np.mean(errors))

        fpr, tpr, _ = roc_curve(true_labels, y_pred)
        roc_auc = auc(fpr, tpr)

        lstm_ae_results.append({
            'Machine_Type': machine_type,
            'Machine_ID': m_id,
            'AUC_Score': roc_auc
        })

        plt.plot(fpr, tpr, label=f'ROC curve (area = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], 'k--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC - {machine_type} with ID {m_id}')
        plt.legend(loc="lower right")
        plt.grid(alpha=0.3)
        
        plt.savefig(f'../results/lstm_ae/{machine_type}_{m_id}_roc.png')
        plt.close()

--- (LSTM AE) Processing fan with ID: 00 ---
Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 9s 152ms/step - loss: 0.8146 - val_loss: 0.5893
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 69ms/step - loss: 0.5560 - val_loss: 0.4896
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 67ms/step - loss: 0.5032 - val_loss: 0.4578
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 69ms/step - loss: 0.4766 - val_loss: 0.4388
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 69ms/step - loss: 0.4581 - val_loss: 0.4269
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - loss: 0.4466 - val_loss: 0.4137
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 68ms/step - loss: 0.4369 - val_loss: 0.4056
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - loss: 0.4276 - val_loss: 0.4007

In [6]:
results_df = pd.DataFrame(lstm_ae_results)
results_df

,Machine_Type,Machine_ID,AUC_Score
0,fan,00,0.564668
1,fan,02,0.752033
2,fan,04,0.561638
3,fan,06,0.890859
4,valve,00,0.195966
5,valve,02,0.627833
6,valve,04,0.541250
7,valve,06,0.373750
8,pump,00,0.727552
9,pump,02,0.617568
